# ViT-Tiny from scratch: high vs. low bandwidth

This notebook first trains a Vision Transformer **from random initialization** on the two folders in `data/`. During that first stage, nothing is pretrained or frozen and every parameter is optimized. A second, clearly separated stage then freezes the complete ViT body and refines only the final classification head.

The model follows the usual ViT-Tiny shape: 16×16 patches, 192-dimensional tokens, 12 transformer blocks, 3 attention heads, and an MLP ratio of 4. The input stem uses one channel because the source BMP files are grayscale.

> Dataset note: there are only 221 images. This notebook uses the project's authoritative 176/45 train/validation manifest so it evaluates the exact same images as the other replication models. There is no independent test set.

## 1. Install dependencies
Run this once in a fresh environment, then restart the kernel if Jupyter asks you to.

In [ ]:
%pip install -q torch torchvision matplotlib scikit-learn

## 2. Imports and configuration
All experiment settings live here. `IMAGE_SIZE` must be divisible by `PATCH_SIZE`.

In [ ]:
import copy
import csv
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
project_candidates = [
    Path.home() / 'Documents' / 'Paper replication',
    Path.home() / 'Desktop' / 'Paper replication',
]
PROJECT_ROOT = next(
    (path for path in project_candidates if (path / 'data' / 'common_split_manifest.csv').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the Paper replication project in Documents or Desktop.')
DATA_DIR = PROJECT_ROOT / 'data'
SPLIT_MANIFEST = DATA_DIR / 'common_split_manifest.csv'
OUTPUT_DIR = PROJECT_ROOT / 'model_reproductions' / '02_vit_tiny_from_scratch' / 'results'
IMAGE_SIZE = 224
PATCH_SIZE = 16
BATCH_SIZE = 16
NUM_WORKERS = 0  # safest in notebooks; increase to 2-4 if desired
EPOCHS = 30
WARMUP_EPOCHS = 5
PATIENCE = EPOCHS + 1  # run all 30 epochs while still retaining the best checkpoint
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 5e-2
HEAD_EPOCHS = 30
HEAD_PATIENCE = HEAD_EPOCHS + 1  # run all 30 head-only epochs
HEAD_LEARNING_RATE = 1e-4

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

assert IMAGE_SIZE % PATCH_SIZE == 0
print(f'Device: {device}')

## 3. Dataset, conservative augmentation, and shared 80/20 split
The project manifest fixes the exact 176 training and 45 validation filenames used by every replication model. The training view is augmented; the validation view is deterministic. No flips are used because they may change the meaning of scientific images.

In [ ]:
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.90, 1.00), ratio=(0.98, 1.02)),
    transforms.RandomAffine(degrees=3, translate=(0.02, 0.02), scale=(0.98, 1.02)),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5,), std=(0.5,)),
    transforms.RandomErasing(p=0.10, scale=(0.01, 0.04), ratio=(0.5, 2.0)),
])

eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5,), std=(0.5,)),
])

train_view = datasets.ImageFolder(DATA_DIR, transform=train_transform)
eval_view = datasets.ImageFolder(DATA_DIR, transform=eval_transform)
assert train_view.samples == eval_view.samples
class_names = train_view.classes
num_classes = len(class_names)

with SPLIT_MANIFEST.open(newline='') as handle:
    split_by_file = {row['file']: row['split'] for row in csv.DictReader(handle)}

sample_names = [Path(path).name for path, _ in train_view.samples]
assert set(sample_names) == set(split_by_file), 'Dataset and split manifest do not match.'
train_idx = [i for i, name in enumerate(sample_names) if split_by_file[name] == 'train']
val_idx = [i for i, name in enumerate(sample_names) if split_by_file[name] == 'validation']
assert len(train_idx) == 176 and len(val_idx) == 45

train_set = Subset(train_view, train_idx)
val_set = Subset(eval_view, val_idx)

loader_args = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                   pin_memory=(device.type == 'cuda'))
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_set, shuffle=True, generator=generator, **loader_args)
val_loader = DataLoader(val_set, shuffle=False, **loader_args)

def class_counts(indices):
    return {class_names[c]: sum(train_view.targets[i] == c for i in indices)
            for c in range(num_classes)}

print('Classes:', train_view.class_to_idx)
print(f'Total: {len(train_view)} | train: {len(train_set)} | validation: {len(val_set)}')
print('Train:', class_counts(train_idx))
print('Validation:', class_counts(val_idx))

In [ ]:
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, image, label in zip(axes.flat, images[:8], labels[:8]):
    ax.imshow(image.squeeze(0).mul(0.5).add(0.5).clamp(0, 1), cmap='gray')
    ax.set_title(class_names[label.item()])
    ax.axis('off')
plt.tight_layout()

## 4. ViT-Tiny implemented directly in PyTorch
A strided convolution creates non-overlapping patch tokens. A learned class token and learned positional embeddings are followed by 12 pre-normalized transformer blocks.

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, image_size=224, patch_size=16, in_channels=1, embed_dim=192):
        super().__init__()
        self.image_size = image_size
        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        self.projection = nn.Conv2d(in_channels, embed_dim,
                                    kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        if x.shape[-2:] != (self.image_size, self.image_size):
            raise ValueError(f'Expected {self.image_size}x{self.image_size}, got {x.shape[-2:]}')
        return self.projection(x).flatten(2).transpose(1, 2)


class MLP(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, dim=192, num_heads=3, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attention = nn.MultiheadAttention(
            embed_dim=dim, num_heads=num_heads, dropout=dropout, batch_first=True
        )
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim, int(dim * mlp_ratio), dropout)

    def forward(self, x):
        normalized = self.norm1(x)
        attended, _ = self.attention(normalized, normalized, normalized, need_weights=False)
        x = x + attended
        return x + self.mlp(self.norm2(x))


class VisionTransformer(nn.Module):
    def __init__(self, image_size=224, patch_size=16, in_channels=1, num_classes=2,
                 embed_dim=192, depth=12, num_heads=3, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(image_size, patch_size, in_channels, embed_dim)
        n = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n + 1, embed_dim))
        self.pos_drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        self.apply(self._init_weights)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Conv2d):
            nn.init.trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.patch_embed(x)
        cls = self.cls_token.expand(x.shape[0], -1, -1)
        x = self.pos_drop(torch.cat((cls, x), dim=1) + self.pos_embed)
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm(x)[:, 0])

In [ ]:
model = VisionTransformer(
    image_size=IMAGE_SIZE, patch_size=PATCH_SIZE, in_channels=1,
    num_classes=num_classes, embed_dim=192, depth=12, num_heads=3,
    mlp_ratio=4.0, dropout=0.10,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert total_params == trainable_params, 'Some parameters are frozen.'
with torch.no_grad():
    output_shape = model(torch.zeros(2, 1, IMAGE_SIZE, IMAGE_SIZE, device=device)).shape
print(f'Parameters: {total_params:,} (all {trainable_params:,} trainable)')
print('Sanity-check output:', output_shape)

## 5. Full-model training loop
AdamW updates every parameter. The schedule warms up for five epochs and then decays with a cosine curve. The best validation state is restored and saved before the classifier-only stage begins.

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.10)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                              betas=(0.9, 0.999), weight_decay=WEIGHT_DECAY)
warmup = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.10, end_factor=1.0, total_iters=WARMUP_EPOCHS
)
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=max(1, EPOCHS - WARMUP_EPOCHS), eta_min=1e-6
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS]
)
use_amp = device.type == 'cuda'
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

def run_epoch(model, loader, training):
    model.train(training)
    loss_sum = 0.0
    correct = 0
    seen = 0

    for inputs, targets in loader:
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(inputs)
                loss = criterion(logits, targets)
            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()

        loss_sum += loss.item() * targets.size(0)
        correct += (logits.argmax(dim=1) == targets).sum().item()
        seen += targets.size(0)

    return loss_sum / seen, correct / seen

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_state = None
best_val_loss = float('inf')
epochs_without_improvement = 0
start = time.time()

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, training=True)
    val_loss, val_acc = run_epoch(model, val_loader, training=False)
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    improved = val_loss < best_val_loss - 1e-4
    if improved:
        best_val_loss = val_loss
        # Clone onto CPU so the checkpoint is a true snapshot on every backend, including MPS.
        best_state = {
            name: tensor.detach().cpu().clone()
            for name, tensor in model.state_dict().items()
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(f'Epoch {epoch:02d}/{EPOCHS} | lr {current_lr:.2e} | '
          f'train loss {train_loss:.4f}, acc {train_acc:.3f} | '
          f'val loss {val_loss:.4f}, acc {val_acc:.3f}' + (' *' if improved else ''))

    if epochs_without_improvement >= PATIENCE:
        print(f'Early stopping after {epoch} epochs.')
        break

if best_state is None:
    raise RuntimeError('Training did not produce a finite validation checkpoint.')
model.load_state_dict(best_state)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), OUTPUT_DIR / 'best_full_model.pth')
print(f'Loaded best validation state ({best_val_loss:.4f}); elapsed {(time.time()-start)/60:.1f} min.')
print('Saved:', OUTPUT_DIR / 'best_full_model.pth')

## 6. Learning curves and shared validation-set evaluation

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs_ran, history['train_loss'], label='train')
axes[0].plot(epochs_ran, history['val_loss'], label='validation')
axes[0].set(title='Loss', xlabel='Epoch', ylabel='Cross-entropy')
axes[1].plot(epochs_ran, history['train_acc'], label='train')
axes[1].plot(epochs_ran, history['val_acc'], label='validation')
axes[1].set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy', ylim=(0, 1.02))
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
plt.tight_layout()

In [ ]:
@torch.inference_mode()
def predict_loader(model, loader):
    model.eval()
    all_targets, all_predictions, all_probabilities = [], [], []
    for inputs, targets in loader:
        logits = model(inputs.to(device, non_blocking=True))
        probabilities = logits.softmax(dim=1).cpu()
        all_targets.extend(targets.tolist())
        all_predictions.extend(probabilities.argmax(dim=1).tolist())
        all_probabilities.extend(probabilities.tolist())
    return np.array(all_targets), np.array(all_predictions), np.array(all_probabilities)

y_true, y_pred, y_prob = predict_loader(model, val_loader)
validation_accuracy = (y_true == y_pred).mean()
print(f'Validation accuracy: {validation_accuracy:.3f} ({(y_true == y_pred).sum()}/{len(y_true)})\n')
print(classification_report(y_true, y_pred, target_names=class_names, digits=3, zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=class_names, cmap='Blues', colorbar=False
)
plt.title('ViT-Tiny validation confusion matrix')
plt.tight_layout()

## 7. Frozen-body classifier refinement

This stage starts from the best full-model checkpoint above. Every body parameter—including patch embedding, class/position tokens, Transformer blocks, and final normalization—is frozen. Only the existing linear classification head is refined. The body stays in evaluation mode, so its dropout is disabled and its output is fixed for a given input.

The original head is retained unless classifier-only training improves validation accuracy; lower validation loss breaks accuracy ties. This is a head-only refinement (or linear-probe-style stage), not full-model fine-tuning.

In [ ]:
# Preserve the full-model result for a direct before/after comparison.
baseline_y_true = y_true.copy()
baseline_y_pred = y_pred.copy()
baseline_accuracy = validation_accuracy

# Freeze everything, then explicitly enable gradients only for the classifier.
for parameter in model.parameters():
    parameter.requires_grad = False
for parameter in model.head.parameters():
    parameter.requires_grad = True

body_before_refinement = {
    name: tensor.detach().cpu().clone()
    for name, tensor in model.state_dict().items()
    if not name.startswith('head.')
}
head_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert head_trainable == sum(p.numel() for p in model.head.parameters())
print(f'Frozen body parameters: {total_params - head_trainable:,}')
print(f'Trainable classifier parameters: {head_trainable:,}')

In [ ]:
head_criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
head_optimizer = torch.optim.AdamW(
    model.head.parameters(), lr=HEAD_LEARNING_RATE, weight_decay=1e-4
)
head_scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

def run_head_epoch(loader, training):
    # The complete body remains deterministic and frozen, including dropout.
    model.eval()
    if training:
        model.head.train()

    loss_sum = 0.0
    correct = 0
    seen = 0
    for inputs, targets in loader:
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        if training:
            head_optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(inputs)
                loss = head_criterion(logits, targets)
            if training:
                head_scaler.scale(loss).backward()
                head_scaler.unscale_(head_optimizer)
                nn.utils.clip_grad_norm_(model.head.parameters(), max_norm=1.0)
                head_scaler.step(head_optimizer)
                head_scaler.update()

        loss_sum += loss.item() * targets.size(0)
        correct += (logits.argmax(dim=1) == targets).sum().item()
        seen += targets.size(0)

    return loss_sum / seen, correct / seen

In [ ]:
head_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
initial_head_val_loss, initial_head_val_acc = run_head_epoch(val_loader, training=False)
best_head_state = {
    name: tensor.detach().cpu().clone()
    for name, tensor in model.head.state_dict().items()
}
best_head_val_loss = initial_head_val_loss
best_head_val_accuracy = initial_head_val_acc
head_epochs_without_improvement = 0

for epoch in range(1, HEAD_EPOCHS + 1):
    train_loss, train_acc = run_head_epoch(train_loader, training=True)
    val_loss, val_acc = run_head_epoch(val_loader, training=False)

    head_history['train_loss'].append(train_loss)
    head_history['train_acc'].append(train_acc)
    head_history['val_loss'].append(val_loss)
    head_history['val_acc'].append(val_acc)

    improved = (
        val_acc > best_head_val_accuracy
        or (val_acc == best_head_val_accuracy and val_loss < best_head_val_loss - 1e-4)
    )
    if improved:
        best_head_val_loss = val_loss
        best_head_val_accuracy = val_acc
        best_head_state = {
            name: tensor.detach().cpu().clone()
            for name, tensor in model.head.state_dict().items()
        }
        head_epochs_without_improvement = 0
    else:
        head_epochs_without_improvement += 1

    print(f'Head epoch {epoch:02d}/{HEAD_EPOCHS} | '
          f'train loss {train_loss:.4f}, acc {train_acc:.3f} | '
          f'val loss {val_loss:.4f}, acc {val_acc:.3f}' + (' *' if improved else ''))

    if head_epochs_without_improvement >= HEAD_PATIENCE:
        print(f'Head-only early stopping after {epoch} epochs.')
        break

model.head.load_state_dict(best_head_state)
body_unchanged = all(
    torch.equal(body_before_refinement[name], tensor.detach().cpu())
    for name, tensor in model.state_dict().items()
    if not name.startswith('head.')
)
assert body_unchanged, 'A frozen body tensor changed during head refinement.'
torch.save(model.state_dict(), OUTPUT_DIR / 'head_refined_model.pth')
print(f'Loaded best classifier state (validation accuracy {best_head_val_accuracy:.3f}, '
      f'loss {best_head_val_loss:.4f}).')
print('Verified: every body tensor is unchanged.')
print('Saved:', OUTPUT_DIR / 'head_refined_model.pth')

In [ ]:
head_epochs_ran = range(1, len(head_history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(head_epochs_ran, head_history['train_loss'], marker='o', label='train')
axes[0].plot(head_epochs_ran, head_history['val_loss'], marker='o', label='validation')
axes[0].axhline(initial_head_val_loss, color='gray', linestyle='--', label='before refinement')
axes[0].set(title='Head-only loss', xlabel='Head epoch', ylabel='Cross-entropy')
axes[1].plot(head_epochs_ran, head_history['train_acc'], marker='o', label='train')
axes[1].plot(head_epochs_ran, head_history['val_acc'], marker='o', label='validation')
axes[1].axhline(initial_head_val_acc, color='gray', linestyle='--', label='before refinement')
axes[1].set(title='Head-only accuracy', xlabel='Head epoch', ylabel='Accuracy', ylim=(0, 1.02))
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
plt.tight_layout()

# Replace the displayed predictions with the final head-refined predictions.
y_true, y_pred, y_prob = predict_loader(model, val_loader)
refined_accuracy = (y_true == y_pred).mean()
print(f'Before head refinement: {baseline_accuracy:.3f} '
      f'({(baseline_y_true == baseline_y_pred).sum()}/{len(baseline_y_true)})')
print(f'After head refinement:  {refined_accuracy:.3f} '
      f'({(y_true == y_pred).sum()}/{len(y_true)})')
print(f'Accuracy change: {refined_accuracy - baseline_accuracy:+.3f}\n')
print(classification_report(y_true, y_pred, target_names=class_names, digits=3, zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=class_names, cmap='Blues', colorbar=False
)
plt.title('Head-refined ViT-Tiny validation confusion matrix')
plt.tight_layout()

## 8. Independent 30-epoch accuracy comparison

The left panel shows the 30-epoch model-from-scratch run. The right panel independently numbers the 30 classifier-refinement epochs from 1 to 30. The refinement starts from the best scratch checkpoint, freezes the complete ViT body, and uses a new optimizer for the classification head. Each panel labels its own maximum training and validation accuracies.

In [ ]:
train_color = '#1f77b4'
validation_color = '#ff7f0e'
figure, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)

def plot_accuracy_phase(axis, train_values, validation_values, title):
    epochs = list(range(1, len(train_values) + 1))
    axis.plot(epochs, train_values, color=train_color, linewidth=2, label='Training')
    axis.plot(epochs, validation_values, color=validation_color, linewidth=2, label='Validation')

    maximum_points = [
        ('Training', train_values, train_color, -40),
        ('Validation', validation_values, validation_color, 14),
    ]
    for label, values, color, text_offset in maximum_points:
        index = int(np.argmax(values))
        epoch = epochs[index]
        value = values[index]
        axis.scatter(epoch, value, color=color, s=70, zorder=5, edgecolor='white')
        axis.annotate(
            f'Max {label}: {value:.1%}\nEpoch {epoch}',
            xy=(epoch, value), xytext=(0, text_offset), textcoords='offset points',
            ha='center', color=color, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=color),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.9),
        )

    axis.set(title=title, xlabel='Epoch', ylim=(0, 1.08), xlim=(1, len(epochs)))
    axis.grid(alpha=0.25)
    axis.legend(loc='lower right')

plot_accuracy_phase(
    axes[0], history['train_acc'], history['val_acc'],
    'ViT-Tiny trained from scratch (30 epochs)',
)
plot_accuracy_phase(
    axes[1], head_history['train_acc'], head_history['val_acc'],
    'ViT-Tiny fine-tuned (30 epochs)',
)
axes[0].set_ylabel('Accuracy')
scratch_best = max(history['val_acc'])
refined_best = max(head_history['val_acc'])
figure.suptitle(
    f'Best observed validation accuracy: {scratch_best:.1%} → {refined_best:.1%} '
    f'({(refined_best - scratch_best) * 100:+.1f} percentage points)'
)
figure.tight_layout()
figure.savefig(OUTPUT_DIR / 'two_stage_training_curves.png', dpi=220, bbox_inches='tight')
print('Saved:', OUTPUT_DIR / 'two_stage_training_curves.png')

## 9. Inspect final head-refined predictions

In [ ]:
# Inspect individual validation predictions and confidence.
n_show = min(12, len(val_set))
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for ax, subset_pos in zip(axes.flat, range(n_show)):
    image, target = val_set[subset_pos]
    pred = y_pred[subset_pos]
    confidence = y_prob[subset_pos, pred]
    ax.imshow(image.squeeze(0).mul(0.5).add(0.5).clamp(0, 1), cmap='gray')
    color = 'green' if pred == target else 'red'
    ax.set_title(f'true: {class_names[target]}\npred: {class_names[pred]} ({confidence:.0%})', color=color)
    ax.axis('off')
for ax in axes.flat[n_show:]:
    ax.axis('off')
plt.tight_layout()

## Reading the result

- The reported scores are validation accuracies, not independent test accuracies. The same 45 images guide both full-model and head-only checkpoint selection, so the final number may be optimistic.
- The head-only stage verifies tensor-by-tensor that the ViT body did not change. It retains the original head unless accuracy improves, using lower validation loss to break accuracy ties.
- A large train/validation gap means overfitting. First try stronger weight decay/dropout, fewer epochs, or a smaller image/model.
- With only 45 validation images, accuracy has high uncertainty. For paper-quality evidence, repeat training over several seeds or use nested cross-validation.
- Compare against a small CNN or linear baseline. A ViT result is more meaningful when it beats simpler models under exactly the same split.